In [1]:
from datasets import load_from_disk
from openai import AzureOpenAI


/media/research/yrl_aida_users/poddubny/poddubnyy/postgraduate/semtab_serializer/.myenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
import os

# Добавляем корневую директорию проекта в sys.path
sys.path.append(os.path.dirname(os.path.abspath('/media/research/yrl_aida_users/poddubny/poddubnyy/postgraduate/semtab_serializer/tests')))

In [4]:
dataset = load_from_disk('none_filtered_new_dataset/')
dataset

Dataset({
    features: ['id', 'statement', 'label', 'table_caption', 'table_text', 'pandas_code', 'pandas_eval', 'nlsep_query', 'semtab_query'],
    num_rows: 88116
})

In [9]:
dataset[0]

{'id': 0,
 'statement': 'haroldo be mention as a brazil scorer for 2 different game',
 'label': 1,
 'table_caption': '1919 in brazilian football',
 'table_text': 'date#result#score#brazil scorers#competition\nmay 11 , 1919#w#6 - 0#friedenreich (3) , neco (2) , haroldo#south american championship\nmay 18 , 1919#w#3 - 1#heitor , amílcar , millon#south american championship\nmay 26 , 1919#d#2 - 2#neco (2)#south american championship\nmay 29 , 1919#w#1 - 0#friedenreich#south american championship\njune 1 , 1919#d#3 - 3#haroldo , arlindo (2)#taça roberto cherry\n',
 'pandas_code': "df['brazil scorers'].apply(lambda x: 'haroldo' in x).sum() == 2",
 'pandas_eval': 'True',
 'nlsep_query': 'haroldo be mention as a brazil scorer for 2 different game  col : date | result | score | brazil scorers | competition row 1 : may 11 , 1919 | w | 6 - 0 | friedenreich (3) , neco (2) , haroldo | south american championship row 2 : may 18 , 1919 | w | 3 - 1 | heitor , amílcar , millon | south american champio

In [5]:

system_prompt = '''You are a Python expert specializing in pandas. Your task is to translate the
given natural language statement into a single-line pandas expression. This
expression must be valid and executable to verify the truth of the statement
using the provided table. Consider the following:
1. The table is represented as a pandas DataFrame named df.
2. Do not include explanations, comments, or multiline outputs.
3. Ensure the output is concise, correct, and when run outputs either True or
False, and strictly in the following Json Format with a single key "PANDA":
"PANDA": "<your Pandas code>"
'''


In [6]:
print(system_prompt)

You are a Python expert specializing in pandas. Your task is to translate the
given natural language statement into a single-line pandas expression. This
expression must be valid and executable to verify the truth of the statement
using the provided table. Consider the following:
1. The table is represented as a pandas DataFrame named df.
2. Do not include explanations, comments, or multiline outputs.
3. Ensure the output is concise, correct, and when run outputs either True or
False, and strictly in the following Json Format with a single key "PANDA":
"PANDA": "<your Pandas code>"



In [7]:
from openai import OpenAI
def send_message(message,max_tokens=2000,top_p=0.9,temperature=0.5,server_url="http://127.0.0.1:8800/v1",api_key="dummy",
                 model_name='deepseek-ai/deepseek-coder-7b-instruct-v1.5',system_prompt='', stop = ["Observation:","\n\n\n\n","\n \n \n"]):
    client = OpenAI(base_url=server_url, api_key=api_key)
    model_input = [
        { 'role': 'system', 'content': system_prompt},
        { 'role': 'user', 'content': message}
    ]
    try:
        print(f"Generating content with model: {model_name}",)
        
        response = client.chat.completions.create(
            model=model_name,
            messages=model_input,
            temperature=temperature,
            max_tokens=max_tokens,
            top_p=top_p,
            stop = stop
        )
        
        return True, response.choices[0].message.content

    except Exception as e:
        print("Failed to call LLM: " + str(e))
        time.sleep(6)
        if hasattr(e, 'response'):
            error_info = e.response.json()  
            code_value = error_info['error']['code']
        else:
            code_value = "context_length_exceeded"
        print("Retrying ...")
        return False, None

In [8]:
import json
import pandas as pd
from io import StringIO
from tqdm import tqdm

In [ ]:

count = 0
corrects_sep = 0
corrects_sem = 0
for entry in tqdm(dataset):
    total += 1
    df = pd.read_csv(StringIO(entry['table_text']), delimiter='#')
    success,response_sep = send_message(f'{entry["statement"]} {entry["nlsep_query"]}',
                                       system_prompt=system_prompt)
    if success:
        pandas_eval = str(bool(eval(response_sep)))
        if str(bool(entry['label'])) == str(pandas_eval):
            corrects_sep += 1
        else:
            print(response_sep)

    success,response_sem = send_message(f'{entry["statement"]} {entry["semtab_query"]}',
                                       system_prompt=system_prompt)
    if success:
        pandas_eval = str(bool(eval(response_sem)))
        if str(bool(entry['label'])) == str(pandas_eval):
            response_sem += 1
        else:
            print(corrects_sem)

print(corrects, total, corrects / total)

In [76]:
import re

def parse_panda_code(input_string):
    """
    Парсит строку и извлекает код, который находится внутри конструкции "PANDA": <код>
    Поддерживает различные форматы: JSON, простой текст, Markdown
    
    Args:
        input_string (str): Входная строка для парсинга
        
    Returns:
        str: Извлеченный код или пустая строка, если код не найден
    """
    # Сначала попробуем найти JSON объект с PANDA
    json_pattern = r'\{[^{}]*PANDA":\s*(.+?)(?:\n|$)?\}'
    json_match = re.search(json_pattern, input_string, re.DOTALL)
    code = None
    pattern = r'"PANDA":\s*(.+?)(?:\n|$)'
    if json_match:
        code = json_match.group(1).strip()
    else:
        match = re.search(pattern, input_string, re.DOTALL)
        if match:
        # Извлекаем код и убираем лишние пробелы по краям
            code = match.group(1).strip()
        # Заменяем одинарные кавычки внутри строки для корректного парсинга JSON

    # Если JSON не найден или не распарсился, используем старый метод
    # Паттерн для поиска кода после "PANDA": 
    # Ищет "PANDA": за которым следует пробел, затем код до конца строки или до следующего символа
    if code != None:
    # Убираем возможные кавычки вокруг кода
        if code.startswith('"') and code.endswith('"'):
            code = code[1:-1]
        elif code.startswith("'") and code.endswith("'"):
            code = code[1:-1]
            
        return code
    
    return ""

In [81]:
corrects_sep = 0

df = pd.read_csv(StringIO(dataset[0]['table_text']), delimiter='#')
success,response_sep = send_message(f'{dataset[0]["statement"]} {dataset[0]["nlsep_query"]}',
                                   system_prompt=system_prompt)
if success:
    pandas_eval = str(bool(eval(parse_panda_code(response_sep))))
    if str(bool(dataset[0]['label'])) == str(pandas_eval):
        corrects_sep += 1
    else:
        print(response_sep)

success,response_sem = send_message(f'{dataset[0]["statement"]} {dataset[0]["semtab_query"]}',
                                   system_prompt=system_prompt)

Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5
Here is a single-line pandas expression that checks if Haroldo is mentioned as a Brazil scorer for two different games:

```json
"PANDA": "df['brazil scorers'].apply(lambda x: 'haroldo' in x.split(', ')).sum() >= 2"
```

This expression splits the 'brazil scorers' column into a list of scorers for each game, checks if 'haroldo' is in the list, and then sums up the True values to see if he is mentioned as a scorer for at least two games.

Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5


In [42]:
print(response_sep)

```json
"PANDA": "df['brazil scorers'].str.contains('haroldo', na=False).sum() > 1"
```
This code checks if 'haroldo' is mentioned as a Brazil scorer for more than 1 game. It uses the str.contains method to check if 'haroldo' is present in the 'brazil scorers' column and then sums up the number of True values, indicating how many games 'haroldo' scored for Brazil.



In [43]:
dataset[0]['label']

1

In [44]:
str(bool(eval(parse_panda_code(response_sep))))

'True'

In [45]:
parse_panda_code(response_sep)

"df['brazil scorers'].str.contains('haroldo', na=False).sum() > 1"

In [90]:
df['brazil scorers'].apply(lambda x: 'haroldo' in x.split(', ')).sum() >= 2

np.False_

In [82]:
print(response_sem)

```json
"PANDA": "df['brazil scorers'].apply(lambda x: 'haroldo' in x.split(',')).sum() > 1"
```
This code checks if 'haroldo' is mentioned as a scorer for more than one game in the 'brazil scorers' column of the DataFrame.



In [83]:
corrects_sem = 0
if success:
    pandas_eval = str(bool(eval(parse_panda_code(response_sem))))
    if str(bool(dataset[0]['label'])) == str(parse_panda_code(pandas_eval)):
        corrects_sem += 1
    else:
        print(corrects_sem)

0


In [77]:
parse_panda_code(response_sem)

"df['brazil scorers'].str.contains('haroldo', na=False).sum() > 1"

In [65]:
response_sem

'Here is the single-line pandas expression based on the given statement:\n\n```json\n{"PANDA": "df[\'brazil scorers\'].str.contains(\'haroldo\', na=False).sum() > 1"}\n```\n\nThis expression checks if \'haroldo\' is mentioned as a Brazil scorer for more than 1 game in the \'brazil scorers\' column of the DataFrame df.\n'

In [66]:
pattern = r'"PANDA":\s*(.+?)(?:\n|$)'
    
    # Используем поиск с флагом re.DOTALL для работы с многострочным кодом
match = re.search(pattern, '''Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5
Here is a Python pandas code snippet that checks if Haroldo is mentioned as a Brazil scorer for more than one game:
```json
"PANDA": "df['brazil scorers'].str.contains('Haroldo', na=False).sum() > 1"
```
This code uses the pandas `str.contains` method to check if 'Haroldo' is in the 'brazil scorers' column. The `na=False` parameter ensures that NaN values are not considered as matches. The `sum()` method is then used to count the number of True values returned by the `contains` method, which corresponds to the number of times 'Haroldo' is mentioned as a Brazil scorer. The result is checked if it's greater than 1.''',
                  re.DOTALL)

In [67]:
code = match.group(1).strip()

In [68]:
code

'"df[\'brazil scorers\'].str.contains(\'Haroldo\', na=False).sum() > 1"'

In [73]:
match.group()

'"PANDA": "df[\'brazil scorers\'].str.contains(\'Haroldo\', na=False).sum() > 1"\n'

In [85]:
bool(eval(parse_panda_code(response_sem)))

False

In [86]:
bool(dataset[0]['label'])

True

In [89]:
from datasets import load_from_disk
from openai import OpenAI
import sys
import os
import json
import pandas as pd
from io import StringIO
from tqdm import tqdm
import re
# Добавляем корневую директорию проекта в sys.path
sys.path.append(os.path.dirname(os.path.abspath('/media/research/yrl_aida_users/poddubny/poddubnyy/postgraduate/semtab_serializer/tests')))

system_prompt = '''You are a Python expert specializing in pandas. Your task is to translate the
given natural language statement into a single-line pandas expression. This
expression must be valid and executable to verify the truth of the statement
using the provided table. Consider the following:
1. The table is represented as a pandas DataFrame named df.
2. Do not include explanations, comments, or multiline outputs.
3. Ensure the output is concise, correct, and when run outputs either True or
False, and strictly in the following Json Format with a single key "PANDA":
"PANDA": "<your Pandas code>"
'''

def send_message(message,max_tokens=2000,top_p=0.9,temperature=0.5,server_url="http://127.0.0.1:8800/v1",api_key="dummy",
                 model_name='deepseek-ai/deepseek-coder-7b-instruct-v1.5',system_prompt='', stop = ["Observation:","\n\n\n\n","\n \n \n"]):
    client = OpenAI(base_url=server_url, api_key=api_key)
    model_input = [
        { 'role': 'system', 'content': system_prompt},
        { 'role': 'user', 'content': message}
    ]
    try:
        print(f"Generating content with model: {model_name}",)
        
        response = client.chat.completions.create(
            model=model_name,
            messages=model_input,
            temperature=temperature,
            max_tokens=max_tokens,
            top_p=top_p,
            stop = stop
        )
        
        return True, response.choices[0].message.content

    except Exception as e:
        print("Failed to call LLM: " + str(e))
        time.sleep(6)
        if hasattr(e, 'response'):
            error_info = e.response.json()  
            code_value = error_info['error']['code']
        else:
            code_value = "context_length_exceeded"
        print("Retrying ...")
        return False, None

def parse_panda_code(input_string):
    """
    Парсит строку и извлекает код, который находится внутри конструкции "PANDA": <код>
    Поддерживает различные форматы: JSON, простой текст, Markdown
    
    Args:
        input_string (str): Входная строка для парсинга
        
    Returns:
        str: Извлеченный код или пустая строка, если код не найден
    """
    # Сначала попробуем найти JSON объект с PANDA
    json_pattern = r'\{[^{}]*PANDA":\s*(.+?)(?:\n|$)?\}'
    json_match = re.search(json_pattern, input_string, re.DOTALL)
    code = None
    pattern = r'"PANDA":\s*(.+?)(?:\n|$)'
    if json_match:
        code = json_match.group(1).strip()
    else:
        match = re.search(pattern, input_string, re.DOTALL)
        if match:
        # Извлекаем код и убираем лишние пробелы по краям
            code = match.group(1).strip()
        # Заменяем одинарные кавычки внутри строки для корректного парсинга JSON

    # Если JSON не найден или не распарсился, используем старый метод
    # Паттерн для поиска кода после "PANDA": 
    # Ищет "PANDA": за которым следует пробел, затем код до конца строки или до следующего символа
    if code != None:
    # Убираем возможные кавычки вокруг кода
        if code.startswith('"') and code.endswith('"'):
            code = code[1:-1]
        elif code.startswith("'") and code.endswith("'"):
            code = code[1:-1]
            
        return code
    
    return ""
def dataset_processing(entry):
    df = pd.read_csv(StringIO(entry['table_text']), delimiter='#')
    success,response_sep = send_message(f'{entry["statement"]} {entry["nlsep_query"]}',
                                       system_prompt=system_prompt)
    if success:
        entry['sep_answ'] = response_sep
        pandas_eval = str(bool(eval(parse_panda_code(response_sep))))
        entry['sep_label'] = str(pandas_eval)
    else:
        entry['sep_answ'] = 'None'
        entry['sep_label'] = 'None'
        
    success,response_sem = send_message(f'{entry["statement"]} {entry["semtab_query"]}',
                                       system_prompt=system_prompt)
    if success:
        entry['sem_answ'] = response_sem
        pandas_eval = str(bool(eval(parse_panda_code(response_sem))))
        entry['sem_label'] = str(pandas_eval)
    else:
        entry['sem_answ'] = 'None'
        entry['sem_label'] = 'None'
dataset = load_from_disk('none_filtered_new_dataset/')
dataset2 = dataset.map(dataset_processing)


Map (num_proc=2):   0%|          | 0/88116 [00:00<?, ? examples/s]

Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5
Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5
Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5
Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5


Map (num_proc=2):   0%|          | 1/88116 [00:02<57:47:52,  2.36s/ examples]

Generating content with model: deepseek-ai/deepseek-coder-7b-instruct-v1.5


Map (num_proc=2):   0%|          | 1/88116 [00:03<81:29:05,  3.33s/ examples]


AttributeError: 'numpy.int64' object has no attribute 'eq'

In [97]:
type(success)

bool

In [124]:
process_data = load_from_disk('answer_gen_semtab_none_filtered_new_dataset')
process_data

Dataset({
    features: ['id', 'statement', 'label', 'table_caption', 'table_text', 'pandas_code', 'pandas_eval', 'nlsep_query', 'semtab_query', 'semtab_answ', 'semtab_label'],
    num_rows: 88116
})

In [127]:
dd = process_data.filter(lambda x: True if x['semtab_answ']!='None' else False,num_proc=17)
dd

Dataset({
    features: ['id', 'statement', 'label', 'table_caption', 'table_text', 'pandas_code', 'pandas_eval', 'nlsep_query', 'semtab_query', 'semtab_answ', 'semtab_label'],
    num_rows: 88116
})

In [128]:
dd = process_data.filter(lambda x: True if x['semtab_label']!='None' else False,num_proc=17)
dd

Filter (num_proc=17): 100%|██████████| 88116/88116 [00:00<00:00, 151805.97 examples/s]


Dataset({
    features: ['id', 'statement', 'label', 'table_caption', 'table_text', 'pandas_code', 'pandas_eval', 'nlsep_query', 'semtab_query', 'semtab_answ', 'semtab_label'],
    num_rows: 62012
})

In [129]:
dd.shape[0]/process_data.shape[0]*100

70.37541422670117

In [130]:
true = dd.filter(lambda x : True if x['semtab_label']==str(bool(x['label'])) else False,num_proc=17)

Filter (num_proc=17): 100%|██████████| 62012/62012 [00:00<00:00, 98299.48 examples/s]


In [132]:
true.shape[0]/process_data.shape[0]*100


40.90176585410141

In [131]:
true.shape[0]/dd.shape[0]*100


58.11939624588789

In [134]:
process_data = load_from_disk('answer_gen_nlsep_none_filtered_new_dataset/')
process_data

Dataset({
    features: ['id', 'statement', 'label', 'table_caption', 'table_text', 'pandas_code', 'pandas_eval', 'nlsep_query', 'semtab_query', 'nlsep_answ', 'nlsep_label'],
    num_rows: 88116
})

In [135]:
dd = process_data.filter(lambda x: True if x['nlsep_answ']!='None' else False,num_proc=17)
dd

Filter (num_proc=17): 100%|██████████| 88116/88116 [00:00<00:00, 156255.01 examples/s]


Dataset({
    features: ['id', 'statement', 'label', 'table_caption', 'table_text', 'pandas_code', 'pandas_eval', 'nlsep_query', 'semtab_query', 'nlsep_answ', 'nlsep_label'],
    num_rows: 88007
})

In [136]:
dd = process_data.filter(lambda x: True if x['nlsep_label']!='None' else False,num_proc=17)
dd

Filter (num_proc=17): 100%|██████████| 88116/88116 [00:00<00:00, 162598.23 examples/s]


Dataset({
    features: ['id', 'statement', 'label', 'table_caption', 'table_text', 'pandas_code', 'pandas_eval', 'nlsep_query', 'semtab_query', 'nlsep_answ', 'nlsep_label'],
    num_rows: 59619
})

In [137]:
dd.shape[0]/process_data.shape[0]*100

67.65967588179218

In [139]:
true = dd.filter(lambda x : True if x['nlsep_label']==str(bool(x['label'])) else False,num_proc=17)

Filter (num_proc=17): 100%|██████████| 59619/59619 [00:00<00:00, 99274.44 examples/s] 


In [140]:
true.shape[0]/dd.shape[0]*100


62.41298914775491

In [141]:
true.shape[0]/process_data.shape[0]*100


42.228426165509106

In [150]:
process_data = load_from_disk('answer_gen_proto_none_filtered_new_dataset/')
process_data

Dataset({
    features: ['id', 'statement', 'label', 'table_caption', 'table_text', 'pandas_code', 'pandas_eval', 'nlsep_query', 'semtab_query', 'proto_query', 'proto_answ', 'proto_label'],
    num_rows: 88116
})

In [151]:
dd = process_data.filter(lambda x: True if x['proto_answ']!='None' else False,num_proc=17)
dd

Dataset({
    features: ['id', 'statement', 'label', 'table_caption', 'table_text', 'pandas_code', 'pandas_eval', 'nlsep_query', 'semtab_query', 'proto_query', 'proto_answ', 'proto_label'],
    num_rows: 88065
})

In [152]:
dd = process_data.filter(lambda x: True if x['proto_label']!='None' else False,num_proc=17)
dd

Dataset({
    features: ['id', 'statement', 'label', 'table_caption', 'table_text', 'pandas_code', 'pandas_eval', 'nlsep_query', 'semtab_query', 'proto_query', 'proto_answ', 'proto_label'],
    num_rows: 46593
})

In [153]:
dd.shape[0]/process_data.shape[0]*100

52.876889554677916

In [154]:
true = dd.filter(lambda x : True if x['proto_label']==str(bool(x['label'])) else False,num_proc=17)

In [155]:
true.shape[0]/process_data.shape[0]*100


31.886377048436152

In [131]:
true.shape[0]/dd.shape[0]*100


58.11939624588789